In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
import os
import re

STATES = {
    "8": "RAJASTHAN"
  
   
    
    
#     "35": "ANDAMAN AND NICOBAR ISLANDS", "28": "ANDHRA PRADESH", "12": "ARUNACHAL PRADESH", 
#     "18": "ASSAM", "10": "BIHAR", "4": "CHANDIGARH", "22": "CHHATTISGARH", "7": "DELHI", 
#     "30": "GOA", "24": "GUJARAT", "6": "HARYANA", "2": "HIMACHAL PRADESH", "1": "JAMMU AND KASHMIR", 
#     "20": "JHARKHAND", "29": "KARNATAKA", "32": "KERALA", "37": "LADAKH", "31": "LAKSHADWEEP", 
#     "23": "MADHYA PRADESH", "27": "MAHARASHTRA", "14": "MANIPUR", "17": "MEGHALAYA", "15": "MIZORAM", 
#     "13": "NAGALAND", "21": "ODISHA", "34": "PUDUCHERRY", "3": "PUNJAB", "8": "RAJASTHAN", 
#     "11": "SIKKIM", "33": "TAMIL NADU", "36": "TELANGANA", 
#     "38": "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "16": "TRIPURA", "5": "UTTARAKHAND", 
#     "9": "UTTAR PRADESH", "19": "WEST BENGAL"
}
Fn = "SEIAA_RAJASTHAN.xlsx"

ACTIVITIES = {
    "29": "7(a) Airports", 
    "15": "4(c) Asbestos milling / asbestos-based products",
    "33": "7(da) Bio-Medical Waste Treatment Facilities", 
    "39": "8(a) Building / Construction",
    "11": "3(b) Cement plants", 
    "19": "5(a) Chemical fertilizers", 
    "16": "4(d) Chlor-alkali industry",
    "14": "4(b)(ii) Coaltar processing units", 
    "8": "2(a) Coal washeries", 
    "13": "4(b) Coke oven plants",
    "37": "7(h) Common Effluent Treatment Plants (CETPs)",
    "32": "7(d ) Common hazardous waste treatment, storage and disposal facilities (TSDFs)",
    "34": "7(i) Common Municipal Solid Waste Management Facility (CMSWMF)", 
    "25": "5(g) Distilleries",
    "75": "5(ga) Grain based distilleries", 
    "31": "7(c ) Industrial estates/ parks/ complexes/ areas, export processing Zones (EPZs), Special Economic Zones",
    "26": "5(h) Integrated paint industry", 
    "22": "5(d) Manmade fibers manufacturing",
    "10": "3(a) Metallurgical Industries (ferrous and non ferrous)", 
    "9": "2(b) Mineral beneficiation",
    "1": "1(a) Mining of minerals", 
    "7": "1(e) Nuclear power projects and processing of nuclear fuel",
    "3": "1(b) Off-shore and onshore oil and gas exploration, development and production",
    "79": "2(c) Pellet Plant", 
    "20": "5(b) Pesticides industry and pesticide specific intermediates (excluding formulations)",
    "21": "5(c) Petro-chemical complexes (industries based on processing of petroleum fractions",
    "23": "5(e ) Petroleum products and petrochemical based processing such as production of carbon black and electrode grade graphite (processes other than cracking",
    "12": "4(a) Petroleum refining industry", 
    "2": "6(a) Pipelines", 
    "35": "7(e) Ports, harbors, breakwaters, dredging",
    "27": "5(i) Pulp & Paper Industry", 
    "5": "1(c) River Valley/Irrigation projects", 
    "36": "7(f) Road",
    "30": "7(b) Ship breaking yards including ship breaking units", 
    "18": "4(f) Skin/hide processing including the tanning industry",
    "77": "1(a)(ii) Slurry pipelines passing through national parks / sanctuaries / coral reefs, ecologically sensitive areas",
    "17": "4(e) Soda ash Industry", 
    "28": "5(j) Sugar Industry", 
    "24": "5(f) Synthetic organic chemicals industry",
    "6": "1(d) Thermal Power Plants", 
    "40": "8(b) Townships/ Area Development Projects / Rehabilitation Centres"
}

driver = webdriver.Chrome()
driver.get("https://parivesh.nic.in/newupgrade/#/trackYourProposal/")
wait = WebDriverWait(driver, 20)

# Click Advance Search
advance_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Show Advance Search')]")))
driver.execute_script("arguments[0].click();", advance_btn)

# Helper function to trigger Angular events
def trigger_angular_select(element, value=None, by_index=None, by_text=None):
    select_obj = Select(element)
    
    if value is not None:
        try:
            select_obj.select_by_value(str(value))
        except NoSuchElementException:
            # Fallback to text or index if value matching fails
            if by_text:
                select_obj.select_by_visible_text(by_text)
            elif by_index is not None:
                select_obj.select_by_index(by_index)
    elif by_text is not None:
        select_obj.select_by_visible_text(by_text)
    elif by_index is not None:
        select_obj.select_by_index(by_index)

    driver.execute_script(
        "arguments[0].dispatchEvent(new Event('change', { bubbles: true }));"
        "arguments[0].dispatchEvent(new Event('input', { bubbles: true }));",
        element
    )

# -----------------------------
# Select Major Clearance Type (Robust Selection)
# -----------------------------
major_clearance_elem = wait.until(
    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='majorClearanceType']"))
)

# Wait until option elements are populated inside the dropdown
wait.until(lambda d: len(Select(major_clearance_elem).options) > 1)

try:
    trigger_angular_select(major_clearance_elem, value="1", by_index=1, by_text="Environment Clearance")
except Exception as e:
    # Fallback to direct index selection
    Select(major_clearance_elem).select_by_index(1)

# Select Issue Authority (SEIAA)
issue_auth_elem = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "select[formcontrolname='issueAuthority']"))
)
trigger_angular_select(issue_auth_elem, value="SEIAA", by_text="SEIAA")

all_table_data = []
headers = []  

# -----------------------------
# Nested Loops
# -----------------------------
for state_value, state_name in STATES.items():
    print(f"\n🌍 Processing State: {state_name} (Value: {state_value})...")
    
    for act_value, act_desc in ACTIVITIES.items():
        print(f"  └── ⚙️ Searching Activity ID: {act_value} ({act_desc[:30]}...)")
        
        # Retry loop for state/activity dropdowns
        selection_success = False
        for attempt in range(3):
            try:
                state_dropdown = wait.until(
                    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
                )
                trigger_angular_select(state_dropdown, value=state_value)
                time.sleep(0.3)
                
                activity_dropdown = wait.until(
                    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='activityId']"))
                )
                trigger_angular_select(activity_dropdown, value=act_value)
                time.sleep(0.3)
                
                selection_success = True
                break
            except (StaleElementReferenceException, TimeoutException):
                time.sleep(0.5)

        if not selection_success:
            print(f"  ❌ Failed to set selection for Activity {act_value}. Skipping...")
            continue

        search_button = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
        )
        
        existing_tables = driver.find_elements(By.ID, "excel-table")
        old_table = existing_tables[0] if existing_tables else None

        driver.execute_script("arguments[0].click();", search_button)
        
        if old_table:
            try:
                wait.until(EC.staleness_of(old_table))
            except Exception:
                time.sleep(1) 

        try:
            wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
            time.sleep(0.5)
        except TimeoutException:
            print(f"  ℹ️ No results found for State: {state_value} + Activity: {act_value}. Proceeding...")
            continue

        # -----------------------------
        # PAGINATION
        # -----------------------------
        page_num = 1
        while True:
            rows_elements = driver.find_elements(By.XPATH, "//table[@id='excel-table']/tbody/tr")
            
            if not rows_elements:
                break
                
            first_row_text = rows_elements[0].text.lower()
            if "no record" in first_row_text or "no data" in first_row_text:
                break

            if not headers:
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                table = soup.find("table", {"id": "excel-table"})
                if table and table.find("thead"):
                    for th in table.find("thead").find_all("th"):
                        headers.append(th.get_text(strip=True))

            total_rows = len(rows_elements)
            print(f"  📊 Page {page_num}: Scraping {total_rows} records...")

            for row in rows_elements:
                try:
                    cols = row.find_elements(By.TAG_NAME, "td")
                    row_data = [col.text.strip() for col in cols]
                    
                    if not row_data or len(row_data) <= 1:
                        continue
                    
                    row_data.append(state_name)
                    row_data.append(act_desc)
                    all_table_data.append(row_data)
                except Exception:
                    continue

            # Pagination control
            try:
                next_buttons = driver.find_elements(By.XPATH, "//button[@aria-label='Next page']")
                if not next_buttons:
                    break
                    
                next_btn = next_buttons[0]
                
                is_disabled = (
                    next_btn.get_attribute("disabled") in ["true", "disabled", True] or
                    next_btn.get_attribute("aria-disabled") == "true" or
                    "mat-button-disabled" in (next_btn.get_attribute("class") or "")
                )
                
                if is_disabled:
                    print(f"  🎉 Reached final page ({page_num}) for Activity {act_value}.")
                    break

                current_signature = rows_elements[0].text if rows_elements else ""

                driver.execute_script("arguments[0].click();", next_btn)
                page_num += 1

                def wait_for_page_transition(d):
                    try:
                        new_rows = d.find_elements(By.XPATH, "//table[@id='excel-table']/tbody/tr")
                        if not new_rows:
                            return False
                        return new_rows[0].text != current_signature
                    except (StaleElementReferenceException, NoSuchElementException):
                        return False

                WebDriverWait(driver, 20).until(wait_for_page_transition)
                time.sleep(0.5)

            except TimeoutException:
                print(f"  ⚠️ Timeout on page {page_num}. Moving to next activity...")
                break
            except Exception as ex:
                print(f"  ⚠️ Pagination exception on page {page_num}: {ex}")
                break

        time.sleep(0.3)


# -----------------------------
# Save Results
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])
    
    if len(headers) < expected_header_count:
        headers.append("State_Name")
    if len(headers) < expected_header_count:
        headers.append("Activity Description")
        
    df = pd.DataFrame(all_table_data, columns=headers[:expected_header_count])
    
    # -------------------------------------------------------------
    # CLEANUP: Remove illegal Excel control characters (\x00-\x08, \x0B-\x0C, \x0E-\x1F)
    # -------------------------------------------------------------
    illegal_xml_chars_re = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]')
    
    df = df.map(lambda x: illegal_xml_chars_re.sub('', x) if isinstance(x, str) else x)

    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head()) 

    output_dir = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Silver"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    file_path = os.path.join(output_dir, Fn)
    df.to_excel(file_path, index=False)
    print(f"\n✅ Total {len(df)} rows scraped and saved to {file_path}")
else:
    print("\n❌ Automation complete. Zero data entries found across the state/activity matrix.")

driver.quit()


🌍 Processing State: RAJASTHAN (Value: 8)...
  └── ⚙️ Searching Activity ID: 29 (7(a) Airports...)
  📊 Page 1: Scraping 5 records...
  🎉 Reached final page (1) for Activity 29.
  └── ⚙️ Searching Activity ID: 15 (4(c) Asbestos milling / asbest...)
  ℹ️ No results found for State: 8 + Activity: 15. Proceeding...
  └── ⚙️ Searching Activity ID: 33 (7(da) Bio-Medical Waste Treatm...)
  📊 Page 1: Scraping 7 records...
  🎉 Reached final page (1) for Activity 33.
  └── ⚙️ Searching Activity ID: 39 (8(a) Building / Construction...)
  📊 Page 1: Scraping 10 records...
  📊 Page 2: Scraping 10 records...
  📊 Page 3: Scraping 10 records...
  📊 Page 4: Scraping 10 records...
  📊 Page 5: Scraping 10 records...
  📊 Page 6: Scraping 10 records...
  📊 Page 7: Scraping 10 records...
  📊 Page 8: Scraping 10 records...
  📊 Page 9: Scraping 10 records...
  📊 Page 10: Scraping 10 records...
  📊 Page 11: Scraping 10 records...
  📊 Page 12: Scraping 10 records...
  📊 Page 13: Scraping 10 records...
  📊 Page 1